PUNTO 1:

Las personas desocupadas se identifican como personas que no están ocupadas actualmente pero que buscan un empleo 
de manera activa. 

Fuente: https://www.indec.gob.ar/uploads/informesdeprensa/mercado_trabajo_eph_4trim24083C6B9E41.pdf Página 6. 

PUNTO 2:

Punto a)

Elegimos trabajar con los datos de Gran Buenos Aires.
En las siguientes celdas eliminamos los datos que no corresponden a la región elegida y unificamos ambas bases.

In [1]:
import pandas as pd

# Cargamos las bases
t2004 = pd.read_stata("C:/Users/User/Downloads/Individual_t104.dta")
t2024 = pd.read_excel("C:/Users/User/Downloads/usu_individual_T124.xlsx")

# Convertimos los nombres de columnas a mayúsculas
t2004.columns = t2004.columns.str.upper()
t2024.columns = t2024.columns.str.upper()

# Añadimos columna de año
t2004['ANO_EPH'] = 2004
t2024['ANO_EPH'] = 2024

# Filtramos por región (Gran Buenos Aires en la base de datos del 2004 y "1" en la base del 2024)
t2004 = t2004[t2004['REGION'] == 'Gran Buenos Aires']
t2024 = t2024[t2024['REGION'] == 1]

# Columnas comunes
columnas_comunes = set(t2004.columns).intersection(t2024.columns)

# Recortamos a las columnas comunes
t2004_recortada = t2004[list(columnas_comunes)].copy()
t2024_recortada = t2024[list(columnas_comunes)].copy()

# Reafirmamos columna de año
t2004_recortada['ANO_EPH'] = 2004
t2024_recortada['ANO_EPH'] = 2024

# Eliminamos columnas completamente vacías
t2004_recortada = t2004_recortada.dropna(axis=1, how='all')
t2024_recortada = t2024_recortada.dropna(axis=1, how='all')

# Unificamos
df_unificada = pd.concat([t2004_recortada, t2024_recortada], ignore_index=True)

# Creamos copia filtrada por GBA (ya está filtrado, pero útil por claridad)
df_gba = df_unificada.copy()

Punto b)

La variables elegidas son las siguientes:
sexo, edad, estado_civil, cobertura_medica, sabe_leer_escribir,
asistencia_escolar, nivel_educativo, finalizo_nivel, condicion_actividad,
categoria_ocupacional, horas_ocupacion_principal, quiere_mas_horas,
busco_mas_horas, intensidad_ocupacional, ponderador

(Renombramos las variables para facilitar la lectura y comprensión de los datos a trabajar.

Las que más valores faltantes tienen dentro de las 15 variables que elegimos son:
- horas_ocupacion_principal
- quiere_mas_horas
- busco_mas_horas
- intensidad_ocupacional
Todas con una cantidad de 3827 datos y correspondientes a la base del primer trimestre de 2024.

La base de datos correspondiente al primer trimestre del 2004 no contiende datos faltantes en las variables seleccionadas.

In [2]:
# Lista de variables seleccionadas
variables = [
    'CH04', 'CH06', 'CH07', 'CH08', 'CH09',
    'CH10', 'CH12', 'CH13', 'ESTADO', 'CAT_OCUP',
    'PP3E_TOT', 'PP03G', 'PP03I', 'INTENSI', 'PONDERA'
]

# Renombramiento de variables seleccionadas
renombrar_vars = {
    'CH04': 'sexo',
    'CH06': 'edad',
    'CH07': 'estado_civil',
    'CH08': 'cobertura_medica',
    'CH09': 'sabe_leer_escribir',
    'CH10': 'asistencia_escolar',
    'CH12': 'nivel_educativo',
    'CH13': 'finalizo_nivel',
    'ESTADO': 'condicion_actividad',
    'CAT_OCUP': 'categoria_ocupacional',
    'PP3E_TOT': 'horas_ocupacion_principal',
    'PP03G': 'quiere_mas_horas',
    'PP03I': 'busco_mas_horas',
    'INTENSI': 'intensidad_ocupacional',
    'PONDERA': 'ponderador'
}

# Llamamos a las variables que fueron renombradas
variables = [
    'sexo', 'edad', 'estado_civil', 'cobertura_medica', 'sabe_leer_escribir',
    'asistencia_escolar', 'nivel_educativo', 'finalizo_nivel', 'condicion_actividad',
    'categoria_ocupacional', 'horas_ocupacion_principal', 'quiere_mas_horas',
    'busco_mas_horas', 'intensidad_ocupacional', 'ponderador'
]

# Creamos una copia explícita del DataFrame filtrado antes de renombrar
df_gba = df_gba.copy()

# Renombramos columnas con seguridad
df_gba.rename(columns=renombrar_vars, inplace=True)

# Los dos pasos anteriores garantizan el buen funcionamiento de los Pandas.

# Creamos tablas de valores faltantes por año
faltantes_2004 = df_gba[df_gba['ANO_EPH'] == 2004][variables].isna().sum()
faltantes_2024 = df_gba[df_gba['ANO_EPH'] == 2024][variables].isna().sum()

# Mostramos los resultados como DataFrames ordenados
tabla_2004 = pd.DataFrame({'Variable': faltantes_2004.index, 'Faltantes_2004': faltantes_2004.values})
tabla_2024 = pd.DataFrame({'Variable': faltantes_2024.index, 'Faltantes_2024': faltantes_2024.values})

# Mostramos las tablas
print("# Tabla de faltantes - Año 2004:")
print(tabla_2004.sort_values(by='Faltantes_2004', ascending=False))

print("\n# Tabla de faltantes - Año 2024:")
print(tabla_2024.sort_values(by='Faltantes_2024', ascending=False))

# Tabla de faltantes - Año 2004:
                     Variable  Faltantes_2004
0                        sexo               0
1                        edad               0
2                estado_civil               0
3            cobertura_medica               0
4          sabe_leer_escribir               0
5          asistencia_escolar               0
6             nivel_educativo               0
7              finalizo_nivel               0
8         condicion_actividad               0
9       categoria_ocupacional               0
10  horas_ocupacion_principal               0
11           quiere_mas_horas               0
12            busco_mas_horas               0
13     intensidad_ocupacional               0
14                 ponderador               0

# Tabla de faltantes - Año 2024:
                     Variable  Faltantes_2024
10  horas_ocupacion_principal            3827
11           quiere_mas_horas            3827
12            busco_mas_horas            3827
13     intens

Punto c)

Cada una de las variables puede contener valores lógicos y otros que van en contra de la lógica, por ejemplo:
Si analizamos cuantas horas trabajó una persona en una semana los valores nunca pueden ser
menores a cero o mayores a 168 (que son las horas totales que contiene una semana).

Por ende, vamos a eliminar los valores que vayan en contra de la lógica de cada una de las variables que elegimos:
- edad: Eliminar si es negativa o mayor a 110
- horas_ocupacion_principal: Eliminar si es negativa o mayor a 168 (número de horas de una semana)
- nivel_educativo: Eliminar si no está entre 1 y 9 (Valores indicativos de la variable)
- categoria_ocupacional: Eliminar si no está entre 1 y 4 (Valores indicativos de la variable)
- intensidad_ocupacional: Eliminar si no está entre 1 y 4 (Valores indicativos de la variable)
- ponderador: Eliminar si es ≤ 0 
- condicion_actividad: Eliminar si no está entre 0 y 4 (Valores indicativos de la variable)
- sexo: Eliminar si no es 1 (varón) o 2 (mujer)
- estado_civil: Eliminar si no está entre 1 y 5 (Valores indicativos de la variable)
- cobertura_medica: Eliminar si valor no está en el conjunto observado válido de combinaciones
- sabe_leer_escribir: Eliminar si no es 1 (sí) o 2 (no)
- asistencia_escolar: Eliminar si no es 1, 2, 3 (Valores indicativos de la variable)
- finalizo_nivel: Eliminar si no es 1, 2 (Valores indicativos de la variable)
- quiere_mas_horas: Eliminar si no es 1 o 2 (Valores indicativos de la variable)
- busco_mas_horas: Eliminar si no es 1 o 2 (Valores indicativos de la variable)

In [3]:
# Creamos una copia de trabajo limpia
df_limpio = df_gba.copy()

# Edad válida entre 0 y 110
df_limpio = df_limpio[(df_limpio['edad'] >= 0) & (df_limpio['edad'] <= 110)]

# Horas de trabajo válidas entre 0 y 168
df_limpio = df_limpio[(df_limpio['horas_ocupacion_principal'] >= 0) & 
                      (df_limpio['horas_ocupacion_principal'] <= 168)]

# Nivel educativo: entre 1 y 9
df_limpio = df_limpio[df_limpio['nivel_educativo'].between(1, 9)]

# Categoría ocupacional: 1 a 4
df_limpio = df_limpio[df_limpio['categoria_ocupacional'].between(1, 4)]

# Intensidad ocupacional: 1 a 4
df_limpio = df_limpio[df_limpio['intensidad_ocupacional'].between(1, 4)]

# Ponderador: > 0
df_limpio = df_limpio[df_limpio['ponderador'] > 0]

# Condición de actividad: 0 a 4
df_limpio = df_limpio[df_limpio['condicion_actividad'].between(0, 4)]

# Sexo: 1 (varón) o 2 (mujer)
df_limpio = df_limpio[df_limpio['sexo'].isin([1, 2])]

# Estado civil: 1 a 5
df_limpio = df_limpio[df_limpio['estado_civil'].between(1, 5)]

# Cobertura médica: conjunto válido según codificación oficial
valores_validos_cobertura = [1, 2, 3, 4, 9, 12, 13, 23, 123]
df_limpio = df_limpio[df_limpio['cobertura_medica'].isin(valores_validos_cobertura)]

# Saber leer y escribir: 1 o 2
df_limpio = df_limpio[df_limpio['sabe_leer_escribir'].isin([1, 2])]

# Asistencia escolar: 1, 2, 3
df_limpio = df_limpio[df_limpio['asistencia_escolar'].isin([1, 2, 3])]

# Finalizó nivel: 1 o 2
df_limpio = df_limpio[df_limpio['finalizo_nivel'].isin([1, 2])]

# Quiere más horas: 1 o 2
df_limpio = df_limpio[df_limpio['quiere_mas_horas'].isin([1, 2])]

# Buscó más horas: 1 o 2
df_limpio = df_limpio[df_limpio['busco_mas_horas'].isin([1, 2])]

TypeError: '>=' not supported between instances of 'str' and 'int'